In [1]:
# 05 - Aggregate Hourly Demand Per Station
# Week 3 - Day 1: Feature Engineering
# Goal: Convert 7.88M trips → hourly pickup counts per station

In [2]:
import pandas as pd
import numpy as np
import gcsfs
import warnings
warnings.filterwarnings('ignore')

# GCS Configuration
BUCKET = 'bluebikes-demand-predictor-data'
CLEANED_PATH = f'gs://{BUCKET}/processed/cleaned/'
STATIONS_PATH = f'gs://{BUCKET}/metadata/stations/stations.parquet'

# Initialize GCS filesystem
fs = gcsfs.GCSFileSystem()

print('Setup complete!')

Setup complete!


In [4]:
# Load cleaned trip data from GCS
print('Loading 2023 data...')
df_2023 = pd.read_parquet(
    f'{CLEANED_PATH}year=2023/',
    storage_options={"token": "google_default"}
)
print(f'  2023 records: {len(df_2023):,}')

print('Loading 2024 data...')
df_2024 = pd.read_parquet(
    f'{CLEANED_PATH}year=2024/',
    storage_options={"token": "google_default"}
)
print(f'  2024 records: {len(df_2024):,}')

# Combine both years
trips = pd.concat([df_2023, df_2024], ignore_index=True)
print(f'\nTotal records: {len(trips):,}')

Loading 2023 data...
  2023 records: 3,157,456
Loading 2024 data...
  2024 records: 4,724,527

Total records: 7,881,983


In [5]:
# Check columns and data types
print('Columns:', trips.columns.tolist())
print('\nDtypes:')
print(trips.dtypes)
print('\nSample rows:')
trips.head(3)

Columns: ['ride_id', 'rideable_type', 'started_at', 'ended_at', 'start_station_name', 'start_station_id', 'end_station_name', 'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng', 'member_casual', 'trip_duration_seconds', 'trip_duration_minutes', 'start_hour', 'start_day_of_week', 'start_month', 'start_year', 'start_date']

Dtypes:
ride_id                          object
rideable_type                    object
started_at               datetime64[ns]
ended_at                 datetime64[ns]
start_station_name               object
start_station_id                 object
end_station_name                 object
end_station_id                   object
start_lat                       float64
start_lng                       float64
end_lat                         float64
end_lng                         float64
member_casual                    object
trip_duration_seconds             int64
trip_duration_minutes           float64
start_hour                        int32
start_day_of_w

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,trip_duration_seconds,trip_duration_minutes,start_hour,start_day_of_week,start_month,start_year,start_date
0,000021C465B427A9,docked_bike,2023-05-19 19:51:59,2023-05-19 20:12:53,Landmark Center - Brookline Ave at Park Dr,B32015,Congress St at Boston City Hall,D32009,42.343691,-71.102353,42.360418,-71.057522,member,1254,20.900000,15,6,5,2023,2023-05-19
1,0000B9E12E10EFC0,docked_bike,2023-07-28 14:39:23,2023-07-28 14:47:15,30 Dane St,S32023,Central Sq Post Office / Cambridge City Hall a...,M32012,42.381001,-71.104025,42.366426,-71.105495,member,472,7.866667,10,6,7,2023,2023-07-28
2,0000D50D74D08A32,docked_bike,2023-05-06 17:39:57,2023-05-06 17:56:32,Government Center - Cambridge St at Court St,B32032,Union Square - Somerville,S32002,42.359825,-71.059796,42.379648,-71.095405,casual,995,16.583333,13,7,5,2023,2023-05-06


In [6]:
# Floor each timestamp to the hour
# e.g., 2023-04-15 14:37:22 → 2023-04-15 14:00:00
trips['hour'] = trips['started_at'].dt.floor('h')

# Count pickups per station per hour
actual_demand = (
    trips
    .groupby(['start_station_id', 'hour'])
    .size()
    .reset_index(name='demand_count')
)

print(f'Rows with actual pickups: {len(actual_demand):,}')
print(f'Unique stations: {actual_demand["start_station_id"].nunique()}')
print(f'Date range: {actual_demand["hour"].min()} to {actual_demand["hour"].max()}')
print(f'\nDemand stats:')
print(actual_demand['demand_count'].describe())

Rows with actual pickups: 2,576,937
Unique stations: 534
Date range: 2023-04-01 04:00:00 to 2025-01-01 04:00:00

Demand stats:
count    2.576937e+06
mean     3.058663e+00
std      3.321604e+00
min      1.000000e+00
25%      1.000000e+00
50%      2.000000e+00
75%      4.000000e+00
max      1.350000e+02
Name: demand_count, dtype: float64


In [7]:
# Get all unique stations and the full hourly range
all_stations = actual_demand['start_station_id'].unique()
hour_min = actual_demand['hour'].min()
hour_max = actual_demand['hour'].max()
all_hours = pd.date_range(start=hour_min, end=hour_max, freq='h')

print(f'Stations: {len(all_stations)}')
print(f'Hours: {len(all_hours):,} ({hour_min} to {hour_max})')

# Cross join: every station × every hour
stations_df = pd.DataFrame({'start_station_id': all_stations})
hours_df = pd.DataFrame({'hour': all_hours})
complete_grid = stations_df.merge(hours_df, how='cross')

print(f'\nComplete grid: {len(complete_grid):,} rows')
print(f'Actual demand rows: {len(actual_demand):,}')
print(f'Zero-demand rows to fill: {len(complete_grid) - len(actual_demand):,}')
print(f'Sparsity: {(1 - len(actual_demand)/len(complete_grid))*100:.1f}% had zero pickups')

Stations: 534
Hours: 15,385 (2023-04-01 04:00:00 to 2025-01-01 04:00:00)

Complete grid: 8,215,590 rows
Actual demand rows: 2,576,937
Zero-demand rows to fill: 5,638,653
Sparsity: 68.6% had zero pickups


In [8]:
# Left join actual counts onto the complete grid
hourly_demand = complete_grid.merge(
    actual_demand,
    on=['start_station_id', 'hour'],
    how='left'
)

# Fill missing demand with 0 (no pickups that hour)
hourly_demand['demand_count'] = hourly_demand['demand_count'].fillna(0).astype(int)

print(f'Final table: {len(hourly_demand):,} rows')
print(f'Zero-demand rows: {(hourly_demand["demand_count"] == 0).sum():,}')
print(f'Non-zero rows: {(hourly_demand["demand_count"] > 0).sum():,}')

Final table: 8,215,590 rows
Zero-demand rows: 5,638,653
Non-zero rows: 2,576,937


In [10]:
# Extract time components
hourly_demand['date'] = hourly_demand['hour'].dt.date
hourly_demand['year'] = hourly_demand['hour'].dt.year
hourly_demand['month'] = hourly_demand['hour'].dt.month
hourly_demand['day_of_week'] = hourly_demand['hour'].dt.dayofweek  # 0=Monday, 6=Sunday
hourly_demand['hour_of_day'] = hourly_demand['hour'].dt.hour
hourly_demand['is_weekend'] = hourly_demand['day_of_week'].isin([5, 6]).astype(int)

print('Columns now:', hourly_demand.columns.tolist())
print(f'\nSample:')
hourly_demand[['start_station_id', 'hour', 'demand_count', 'day_of_week', 'hour_of_day', 'is_weekend']].head(10)

Columns now: ['start_station_id', 'hour', 'demand_count', 'date', 'year', 'month', 'day_of_week', 'hour_of_day', 'is_weekend']

Sample:


,start_station_id,hour,demand_count,day_of_week,hour_of_day,is_weekend
0,A32000,2023-04-01 04:00:00,0,5,4,1
1,A32000,2023-04-01 05:00:00,0,5,5,1
2,A32000,2023-04-01 06:00:00,0,5,6,1
3,A32000,2023-04-01 07:00:00,0,5,7,1
4,A32000,2023-04-01 08:00:00,0,5,8,1
5,A32000,2023-04-01 09:00:00,0,5,9,1
6,A32000,2023-04-01 10:00:00,0,5,10,1
7,A32000,2023-04-01 11:00:00,0,5,11,1
8,A32000,2023-04-01 12:00:00,0,5,12,1
9,A32000,2023-04-01 13:00:00,0,5,13,1


In [11]:
print('='*60)
print('VALIDATION CHECKS')
print('='*60)

# 1. No duplicate (station, hour) pairs
dupes = hourly_demand.duplicated(subset=['start_station_id', 'hour']).sum()
print(f'\n1. Duplicate (station, hour) pairs: {dupes}')

# 2. No null demand counts
nulls = hourly_demand['demand_count'].isnull().sum()
print(f'2. Null demand counts: {nulls}')

# 3. No negative demand
negatives = (hourly_demand['demand_count'] < 0).sum()
print(f'3. Negative demand counts: {negatives}')

# 4. Total pickups match original trip count
total_demand = hourly_demand['demand_count'].sum()
total_trips = len(trips)
print(f'\n4. Total pickups in demand table: {total_demand:,}')
print(f'   Total trips in original data:  {total_trips:,}')
print(f'   Match: {total_demand == total_trips}')

# 5. Grid completeness
n_stations = hourly_demand['start_station_id'].nunique()
n_hours = hourly_demand['hour'].nunique()
expected = n_stations * n_hours
print(f'\n5. Grid: {n_stations} stations x {n_hours:,} hours = {expected:,} expected')
print(f'   Actual rows: {len(hourly_demand):,}')
print(f'   Complete: {len(hourly_demand) == expected}')

VALIDATION CHECKS

1. Duplicate (station, hour) pairs: 0
2. Null demand counts: 0
3. Negative demand counts: 0

4. Total pickups in demand table: 7,881,983
   Total trips in original data:  7,881,983
   Match: True

5. Grid: 534 stations x 15,385 hours = 8,215,590 expected
   Actual rows: 8,215,590
   Complete: True


In [12]:
# Average demand by hour of day
hourly_pattern = hourly_demand.groupby('hour_of_day')['demand_count'].mean()
print('AVERAGE DEMAND BY HOUR OF DAY')
print('-'*40)
for hour, avg in hourly_pattern.items():
    bar = '█' * int(avg * 5)
    print(f'  {hour:02d}:00  {avg:.2f}  {bar}')

AVERAGE DEMAND BY HOUR OF DAY
----------------------------------------
  00:00  1.10  █████
  01:00  0.82  ████
  02:00  0.64  ███
  03:00  0.45  ██
  04:00  0.27  █
  05:00  0.20  
  06:00  0.12  
  07:00  0.05  
  08:00  0.04  
  09:00  0.12  
  10:00  0.34  █
  11:00  0.82  ████
  12:00  1.50  ███████
  13:00  1.28  ██████
  14:00  1.03  █████
  15:00  1.07  █████
  16:00  1.21  ██████
  17:00  1.25  ██████
  18:00  1.32  ██████
  19:00  1.50  ███████
  20:00  1.89  █████████
  21:00  2.39  ███████████
  22:00  2.10  ██████████
  23:00  1.55  ███████


In [13]:
# Check: what time did the most trips start?
print('Top 10 busiest hours in the dataset:')
print(trips['started_at'].dt.hour.value_counts().head(10))

Top 10 busiest hours in the dataset:
started_at
21    817645
22    719049
20    646159
23    530988
12    512784
19    512154
18    451654
13    438364
17    429429
16    413492
Name: count, dtype: int64


In [15]:
# Convert UTC to Eastern Time
trips['started_at_et'] = trips['started_at'].dt.tz_localize('UTC').dt.tz_convert('US/Eastern')

# Remove timezone info and floor to hour
# Do tz_localize(None) FIRST, then floor — avoids the ambiguous time error
trips['hour'] = trips['started_at_et'].dt.tz_localize(None).dt.floor('h')

# Re-aggregate
actual_demand = (
    trips
    .groupby(['start_station_id', 'hour'])
    .size()
    .reset_index(name='demand_count')
)

print(f'Rows with actual pickups: {len(actual_demand):,}')
print(f'Date range: {actual_demand["hour"].min()} to {actual_demand["hour"].max()}')

Rows with actual pickups: 2,576,937
Date range: 2023-04-01 00:00:00 to 2024-12-31 23:00:00


In [16]:
# Rebuild the complete grid with corrected timestamps
all_stations = actual_demand['start_station_id'].unique()
hour_min = actual_demand['hour'].min()
hour_max = actual_demand['hour'].max()
all_hours = pd.date_range(start=hour_min, end=hour_max, freq='h')

complete_grid = pd.DataFrame({'start_station_id': all_stations}).merge(
    pd.DataFrame({'hour': all_hours}), how='cross'
)

hourly_demand = complete_grid.merge(
    actual_demand,
    on=['start_station_id', 'hour'],
    how='left'
)
hourly_demand['demand_count'] = hourly_demand['demand_count'].fillna(0).astype(int)

print(f'Grid: {len(all_stations)} stations x {len(all_hours):,} hours = {len(hourly_demand):,} rows')
print(f'Total pickups: {hourly_demand["demand_count"].sum():,} (should be 7,881,983)')

Grid: 534 stations x 15,384 hours = 8,215,056 rows
Total pickups: 7,881,983 (should be 7,881,983)


In [17]:
# Re-check hourly pattern with Eastern time
hourly_demand['hour_of_day'] = hourly_demand['hour'].dt.hour
hourly_pattern = hourly_demand.groupby('hour_of_day')['demand_count'].mean()

print('AVERAGE DEMAND BY HOUR OF DAY (Eastern Time)')
print('-'*50)
for hour, avg in hourly_pattern.items():
    bar = '█' * int(avg * 5)
    print(f'  {hour:02d}:00  {avg:.2f}  {bar}')

AVERAGE DEMAND BY HOUR OF DAY (Eastern Time)
--------------------------------------------------
  00:00  0.25  █
  01:00  0.19  
  02:00  0.10  
  03:00  0.04  
  04:00  0.04  
  05:00  0.13  
  06:00  0.38  █
  07:00  0.92  ████
  08:00  1.65  ████████
  09:00  1.19  █████
  10:00  0.98  ████
  11:00  1.08  █████
  12:00  1.24  ██████
  13:00  1.26  ██████
  14:00  1.34  ██████
  15:00  1.53  ███████
  16:00  1.95  █████████
  17:00  2.44  ████████████
  18:00  2.01  ██████████
  19:00  1.46  ███████
  20:00  1.04  █████
  21:00  0.79  ███
  22:00  0.61  ███
  23:00  0.41  ██


In [18]:
# Add time features
hourly_demand['date'] = hourly_demand['hour'].dt.date
hourly_demand['year'] = hourly_demand['hour'].dt.year
hourly_demand['month'] = hourly_demand['hour'].dt.month
hourly_demand['day_of_week'] = hourly_demand['hour'].dt.dayofweek
hourly_demand['is_weekend'] = hourly_demand['day_of_week'].isin([5, 6]).astype(int)

# Validation
total = hourly_demand['demand_count'].sum()
dupes = hourly_demand.duplicated(subset=['start_station_id', 'hour']).sum()
nulls = hourly_demand['demand_count'].isnull().sum()

print(f'Final table: {len(hourly_demand):,} rows')
print(f'Columns: {hourly_demand.columns.tolist()}')
print(f'\nValidation:')
print(f'  Duplicates: {dupes}')
print(f'  Nulls: {nulls}')
print(f'  Total pickups: {total:,} (match: {total == 7881983})')
print(f'  Grid complete: {len(hourly_demand) == 534 * 15384}')

Final table: 8,215,056 rows
Columns: ['start_station_id', 'hour', 'demand_count', 'hour_of_day', 'date', 'year', 'month', 'day_of_week', 'is_weekend']

Validation:
  Duplicates: 0
  Nulls: 0
  Total pickups: 7,881,983 (match: True)
  Grid complete: True


In [19]:
import os

# Sort for clean output
hourly_demand = hourly_demand.sort_values(['start_station_id', 'hour']).reset_index(drop=True)

# Save locally
os.makedirs('data/features', exist_ok=True)
local_path = 'data/features/hourly_demand_by_station.parquet'
hourly_demand.to_parquet(local_path, index=False)
print(f'Saved locally: {local_path}')
print(f'  Size: {os.path.getsize(local_path) / (1024*1024):.1f} MB')

# Upload to GCS
gcs_path = f'gs://{BUCKET}/processed/features/hourly_demand_by_station.parquet'
hourly_demand.to_parquet(
    gcs_path,
    index=False,
    storage_options={"token": "google_default"}
)
print(f'Uploaded to GCS: {gcs_path}')

Saved locally: data/features/hourly_demand_by_station.parquet
  Size: 16.3 MB
Uploaded to GCS: gs://bluebikes-demand-predictor-data/processed/features/hourly_demand_by_station.parquet


In [20]:
print('='*60)
print('NOTEBOOK 05 - AGGREGATE DEMAND: COMPLETE')
print('='*60)
print(f'Input:  7,881,983 individual trips')
print(f'Output: {len(hourly_demand):,} station-hour rows')
print(f'  Stations: 534')
print(f'  Hours: 15,384 (Apr 2023 - Dec 2024, Eastern Time)')
print(f'  Sparsity: 68.6% zero-demand rows')
print(f'  Target: demand_count (hourly pickups per station)')
print(f'\nKey fix: Converted UTC → Eastern Time')
print(f'  Morning peak: 8 AM | Evening peak: 5 PM')
print(f'\nSaved:')
print(f'  Local: data/features/hourly_demand_by_station.parquet (16.3 MB)')
print(f'  GCS: gs://{BUCKET}/processed/features/hourly_demand_by_station.parquet')
print(f'\n→ Next: Notebook 06 - Join weather, stations, holidays + lag features')

NOTEBOOK 05 - AGGREGATE DEMAND: COMPLETE
Input:  7,881,983 individual trips
Output: 8,215,056 station-hour rows
  Stations: 534
  Hours: 15,384 (Apr 2023 - Dec 2024, Eastern Time)
  Sparsity: 68.6% zero-demand rows
  Target: demand_count (hourly pickups per station)

Key fix: Converted UTC → Eastern Time
  Morning peak: 8 AM | Evening peak: 5 PM

Saved:
  Local: data/features/hourly_demand_by_station.parquet (16.3 MB)
  GCS: gs://bluebikes-demand-predictor-data/processed/features/hourly_demand_by_station.parquet

→ Next: Notebook 06 - Join weather, stations, holidays + lag features
